# SBD Model Demo (Request 9)

Демонстрация включает:
- сквозной tracing (`trace_id`, `span_id`, `parent_span_id`);
- сертификацию прошивки (`VendorUAS -> Regulator`);
- регистрацию БАС с форматом `UAS-<TYPE>-<VENDOR_CODE>-<NUMBER>`;
- закупку БАС оператором и приписку к `DronePort`;
- основной e2e-сценарий заказа.

## Инициализация окружения

Импортируем runtime-функции из `sbd_demo.py` и поднимаем все процессы сущностей.

In [ ]:
import sys
import time
import multiprocessing as mp
from pathlib import Path
from typing import Any, Dict, List, Optional
import queue as pyqueue


def _find_repo_root(start: Path) -> Path:
    root = start
    for _ in range(8):
        if (root / "broker").exists() and (root / "systems").exists():
            return root
        if root.parent == root:
            break
        root = root.parent
    return start


repo_root = _find_repo_root(Path.cwd())
sbd_demo_code_dir = repo_root / "demos" / "sbd-model-simple-demo" / "sbd-model-demo-code"
sys.path.insert(0, str(sbd_demo_code_dir))

from actions import (
    PLACE_ORDER,
    REQUEST_FIRMWARE_CERTIFICATION,
    REQUEST_UAS_PURCHASE,
    REQUEST_UAS_REGISTRATION,
)
from sbd_demo import (
    _launch,
    _send_rpc_from_orchestrator,
    _shutdown,
    _wait_for_rpc_main,    
)

from world_state import build_world

world = build_world()
runtime = _launch(world)
out_q = runtime["orchestrator_reply_queue"]
out_name = runtime["orchestrator_reply_name"]
in_q = runtime["broker_in_queue"]

: 

## Сценарий 1: Сертификация прошивки

Проверяем, что цепочка проходит через `VendorUAS` и `Regulator`, а в ответе приходит сертификат.

In [ ]:
cert_corr, cert_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="vendor_uas",
    action=REQUEST_FIRMWARE_CERTIFICATION,
    payload={
        "firmware_version": "2.1.0",
        "vendor_code": "SBD",
        "uas_type": "AGRO",
        "artifacts": ["report.pdf", "tests.json"],
    },
)
cert_msg = _wait_for_rpc_main(out_q, correlation_id=cert_corr, expected_sender="vendor_uas", timeout_s=60.0)
assert cert_msg["trace_id"] == cert_trace
firmware_cert = cert_msg["payload"]["firmware_certification_result"]
assert firmware_cert["approved"] is True
firmware_cert

## Сценарий 2: Регистрация БАС

Проверяем формат `registered_uas_id` и трассировку.

In [ ]:
import re

reg_corr, reg_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="vendor_uas",
    action=REQUEST_UAS_REGISTRATION,
    payload={
        "firmware_version": "2.1.0",
        "vendor_code": "SBD",
        "uas_type": "AGRO",
        "imei": "356938035643809",
    },
)
reg_msg = _wait_for_rpc_main(out_q, correlation_id=reg_corr, expected_sender="vendor_uas", timeout_s=60.0)
assert reg_msg["trace_id"] == reg_trace
registered_uas_id = reg_msg["payload"]["registered_uas_id"]
assert re.fullmatch(r"UAS-[A-Z0-9_]+-[A-Z0-9_]+-\d{6}", registered_uas_id)
registered_uas_id

## Сценарий 3: Закупка БАС Operator и приписка к DronePort

Оператор закупает 2 новые БАС у `VendorUAS`, затем приписывает их к `droneport_B`.

In [ ]:
purchase_corr, purchase_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="operator_1",
    action=REQUEST_UAS_PURCHASE,
    payload={
        "quantity": 2,
        "target_droneport_id": "droneport_B",
        "vendor_code": "SBD",
        "uas_type": "AGRO",
        "firmware_version": "2.1.0",
        "model_id": "agro-model-new",
        "supported_task_types": ["agro"],
        "base_cost": 9800.0,
        "imeis": ["356938035643810", "356938035643811"],
    },
)
purchase_msg = _wait_for_rpc_main(out_q, correlation_id=purchase_corr, expected_sender="operator_1", timeout_s=60.0)
assert purchase_msg["trace_id"] == purchase_trace
purchase_payload = purchase_msg["payload"]
for uas_id in purchase_payload["uas_purchase_result"]["registered_uas_ids"]:
    assert re.fullmatch(r"UAS-[A-Z0-9_]+-[A-Z0-9_]+-\d{6}", uas_id)
assert purchase_payload["attachment"]["droneport_id"] == "droneport_B"
assert purchase_payload["attachment"]["assigned_count"] == 2
purchase_payload

## Сценарий 4: Базовый end-to-end заказ

После новых сценариев запускаем основной поток заказа и проверяем результат посадки.

In [ ]:
order_corr, order_trace = _send_rpc_from_orchestrator(
    broker_in_queue=in_q,
    reply_to=out_name,
    receiver="customer",
    action=PLACE_ORDER,
    payload={
        "order": {
            "id": "ORDER-DEMO-001",
            "scenario_type": "agro",
            "destination": {"lat": 55.75, "lon": 37.61},
            "return_port": "droneport_B",
            "coverage": {
                "min_payload": 3.5,
                "min_range": 1.0,
                "min_battery": 0.8,
            },
        },
        "scenario_security_goals": ["SG_ID_AUTH_001", "SG_ID_SAF_002"],
        "max_price": 999999.0,
    },
)
final_msg = _wait_for_rpc_main(out_q, correlation_id=order_corr, expected_sender="customer", timeout_s=180.0)
assert final_msg["trace_id"] == order_trace
final_payload = final_msg["payload"]
assert final_payload["status"] == "ok"
assert final_payload["order_execution_completed"]["landing_coordinates"] == [55.76, 37.62]
final_payload

## Завершение

Останавливаем процессы и выводим агрегированный результат.

In [ ]:
_shutdown(runtime)
artifacts = {
    "firmware_cert": firmware_cert,
    "registered_uas_id": registered_uas_id,
    "purchased_uas_ids": purchase_payload["uas_purchase_result"]["registered_uas_ids"],
    "final_order_result": final_payload,
}
artifacts